# 04 · Architectures, parameters, and losses

**Goals:** build models with `Sequential` and `nn.Module`, inspect shapes and parameters, and choose a loss compatible with the task. A layer contains learnable parameters; an activation allows a network to represent nonlinear relationships.

In [ ]:
import torch
from torch import nn

torch.manual_seed(486)

model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
inputs = torch.randn(4, 3)  # Four observations, three features each.
labels = torch.tensor([0, 1, 0, 1], dtype=torch.long)

logits = model(inputs)
print(model, "\nlogits shape:", logits.shape)
print("parameter shapes:", [(name, tuple(p.shape)) for name, p in model.named_parameters()])
print("number of trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))


For two-class classification with two output logits, `CrossEntropyLoss` combines the appropriate log-softmax and negative log-likelihood. Pass the raw logits, and use softmax only when you need probabilities.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
print("loss:", loss_fn(logits, labels).item())

print("probabilities:\n", torch.softmax(logits, dim=1))
print("predicted classes:", logits.argmax(dim=1))


A custom module makes it easier to name layers or define more complex forward computations. `forward` should describe the computation; invoking `model(inputs)` calls it through PyTorch's module machinery.

In [ ]:
class Classifier(nn.Module):
    def __init__(self, n_features, hidden_units, n_classes):
        super().__init__()
        self.hidden = nn.Linear(n_features, hidden_units)
        self.output = nn.Linear(hidden_units, n_classes)

    def forward(self, x):
        x = torch.relu(self.hidden(x))
        return self.output(x)

custom_model = Classifier(3, 8, 2)
print("custom output:", custom_model(inputs).shape)

# Regression: one continuous output and matching (batch, 1) float targets.
regression_model = nn.Linear(3, 1)
targets = torch.randn(4, 1)
print("regression MSE:", nn.MSELoss()(regression_model(inputs), targets).item())


## Try it

1. Change the hidden width and calculate the parameter count by hand.

2. Add a second hidden layer to `Classifier` and confirm the output shape.

3. Compare raw logits, predicted classes, and probabilities. Why is softmax not passed into `CrossEntropyLoss`?

4. What goes wrong if regression predictions have shape `(4, 1)` but targets have shape `(4,)`?